In [ ]:
import pandas as pd
import os
from openai import OpenAI
from tqdm import tqdm
from getpass import getpass
import time

# 1. API Setup
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key eingeben: ")
client = OpenAI()

def get_llm_score(reference, candidate):
    """Bewertet die fachliche Qualität auf einer Skala von 1-5."""
    prompt = f"""Du bist Experte für österreichisches Steuerrecht.
Vergleiche die KI-Antwort mit der Referenz-Antwort.

REFERENZ: {reference}
KI-ANTWORT: {candidate}

Bewerte die KI-ANTWORT von 1 bis 5:
1: Fachlich völlig falsch oder Halluzination.
2: Größtenteils falsch, nur Bruchstücke korrekt.
3: Fachlich okay, aber unvollständig oder ungenau.
4: Fachlich korrekt, deckt die wichtigsten Punkte ab.
5: Exzellent, präzise und fachlich absolut sicher.

Antworte NUR mit der Zahl."""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=1
        )
        return int(response.choices[0].message.content.strip())
    except:
        return None

# 2. Daten laden
# Referenz mit Semikolon-Separator
df_ref = pd.read_csv('Dataset_clean_Answers.csv', sep=';')

output_files = {
    "Model_1_Inference": "inference_output.csv",
    "Model_2_FineTune": "fine_tune_output.csv",
    "Model_3_RAG": "Rag_Output.csv"
}

# 3. Prozessschleife
final_stats = []

for label, filename in output_files.items():
    print(f"\n--- Evaluiere {label} ({filename}) ---")

    if not os.path.exists(filename):
        print(f"Datei {filename} nicht gefunden. Überspringe...")
        continue

    df_out = pd.read_csv(filename)

    # Merge über ID
    df_compare = pd.merge(df_out, df_ref[['id', 'correct_answer']], on='id', how='inner')

    scores = []
    for _, row in tqdm(df_compare.iterrows(), total=len(df_compare)):
        score = get_llm_score(row['correct_answer'], row['answer'])
        scores.append(score)

    df_compare['llm_score'] = scores

    # Speichern der Detail-Ergebnisse
    detail_filename = f"LLM_Judge_Detail_{label}.csv"
    df_compare.to_csv(detail_filename, index=False)

    avg_score = sum(filter(None, scores)) / len([s for s in scores if s is not None])
    final_stats.append({"Modell": label, "Avg_LLM_Score": round(avg_score, 4)})

# 4. Endergebnis
df_stats = pd.DataFrame(final_stats)
df_stats.to_csv("Final_Model_Comparison_Scores.csv", index=False)
print("\nVergleich abgeschlossen!")
print(df_stats)

OpenAI API Key eingeben: ··········

--- Evaluiere Model_1_Inference (inference_output.csv) ---


100%|██████████| 643/643 [06:35<00:00,  1.63it/s]



--- Evaluiere Model_2_FineTune (fine_tune_output.csv) ---


100%|██████████| 643/643 [06:49<00:00,  1.57it/s]



--- Evaluiere Model_3_RAG (Rag_Output.csv) ---


100%|██████████| 643/643 [06:29<00:00,  1.65it/s]


Vergleich abgeschlossen!
              Modell  Avg_LLM_Score
0  Model_1_Inference         3.5661
1   Model_2_FineTune         1.9129
2        Model_3_RAG         2.4821
